# Colorectal cancer classification from stool microbial profiles
Cross-cohort associations reported by Wirbel et al. (*Nature Medicine*, 2019) motivate using microbial detection as a predictor of case status. This classifier uses all 849 mOTU columns, age, BMI, sequencing depth, gender and study cohort. All taxa are retained.

Run all cells with the supplied `train.csv` and `test.csv` next to the notebook, in `data/`, or in `CU_hackathon_participants/`. The analysis used scikit-learn 1.6.1. No network connection is required.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

directories = [Path("."), Path("data"), Path("CU_hackathon_participants")]
available = [d for d in directories if all((d / f).is_file() for f in ("train.csv", "test.csv"))]
if len(available) != 1:
    raise ValueError("Place one competition CSV pair in one of the documented locations.")
data_dir = available[0]
train = pd.read_csv(data_dir / "train.csv", dtype={"sample_id": str}).sort_values("sample_id")
taxa = sorted(c for c in train if "mOTU" in c)
numeric = ["age", "bmi", "library_size"]
categorical = ["gender", "cohort"]
features = taxa + numeric + categorical
assert len(train) == 415 and len(taxa) == 849 and set(train.label) == {0, 1}
assert train.sample_id.is_unique and train.sample_id.notna().all()
assert set(train.columns) == set(features + ["sample_id", "label"])

## Frozen model
Microbial values are median-imputed, converted to presence at abundance > 0, and standardized. Numeric metadata use median imputation and standardization. Categorical metadata use mode imputation and one-hot encoding with support for unseen categories.

Development used 332 samples (165 cases and 167 controls). The original 20% holdout contained 83 (40 cases and 43 controls), stratified jointly by cohort and label after sorting sample IDs, with seed 42. All samples were retained. The fixed model achieved probability ROC AUC 0.7494 and balanced accuracy 0.6622 in the single final holdout evaluation. For submission, the pipeline below fits all preprocessing and trees on the complete labeled set.

In [ ]:
def detected(values):
    return (values > 0).astype(float)


microbial = make_pipeline(
    SimpleImputer(strategy="median", keep_empty_features=True),
    FunctionTransformer(detected),
    StandardScaler(),
)
metadata_numeric = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())
metadata_categorical = make_pipeline(
    SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")
)
preprocess = ColumnTransformer([
    ("micro", microbial, taxa),
    ("numeric", metadata_numeric, numeric),
    ("category", metadata_categorical, categorical),
])
model = make_pipeline(
    preprocess,
    ExtraTreesClassifier(n_estimators=160, max_features="sqrt", min_samples_leaf=2,
                         random_state=42, n_jobs=1),
)
with threadpool_limits(limits=1):
    model.fit(train[features], train.label)

## Submission
The table keeps the supplied test order. A case probability of at least 0.50 gives label 1.

In [ ]:
test = pd.read_csv(data_dir / "test.csv", dtype={"sample_id": str})
assert set(test.columns) == set(features + ["sample_id"])
assert len(test) == 160 and test.sample_id.is_unique and test.sample_id.notna().all()
assert model.classes_.tolist() == [0, 1]
with threadpool_limits(limits=1):
    probabilities = model.predict_proba(test[features])[:, 1]
assert np.isfinite(probabilities).all()
submission = test[["sample_id"]].assign(label=(probabilities >= 0.50).astype(int))
display({"text/plain": submission.to_string(index=False), "text/html": submission.to_html(index=False),
         "text/csv": submission.to_csv(index=False)}, raw=True)